<a href="https://colab.research.google.com/github/icervecon/Dividend-Design-UCITS-Funds/blob/main/Paper_Dividend_Design_UCITS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dividend Policies as Product Design: Evidence from UCITS Funds and the Role of Asset Management Firms

In [75]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [76]:
file_path = "ucits_dividend_data.xlsx"
df = pd.read_excel(file_path)

In [77]:
df['dividend'] = df['has_paid_dividends_i']
df['CL4_yes'] = (df['CL4'] == 'Yes').astype(int)

Table 1- Descriptive Statistics- Panel A- Full Sample

In [78]:
vars_desc = [
    'dividend',
    'retail_eligible_i',
    'CL4_yes',
    'log_tna',
    'fund_age_years',
    'is_equity',
    'is_fixed_income'
]

desc = df[vars_desc].describe().T

desc = desc[['mean', 'std', 'min', 'max']]
desc

,mean,std,min,max
dividend,0.310902,0.462905,0.000000,1.000000
retail_eligible_i,0.862805,0.344141,0.000000,1.000000
CL4_yes,0.421005,0.493765,0.000000,1.000000
log_tna,3.929630,2.090602,0.000089,11.677366
fund_age_years,14.159420,8.492873,0.005476,53.188227
is_equity,0.348268,0.476465,0.000000,1.000000
is_fixed_income,0.299111,0.457910,0.000000,1.000000


Table 1- Descriptive Statistics- Panel B- By Retail Eligibility

In [79]:
vars_desc = [
    'dividend',
    'CL4_yes',
    'log_tna',
    'fund_age_years',
    'is_equity',
    'is_fixed_income'
]

grouped = df.groupby('retail_eligible_i')[vars_desc].mean().T

grouped.columns = ['Non-Retail', 'Retail']

grouped['Difference'] = grouped['Retail'] - grouped['Non-Retail']

grouped

,Non-Retail,Retail,Difference
dividend,0.144444,0.409894,0.265450
CL4_yes,0.977778,0.985866,0.008088
log_tna,4.017750,4.184931,0.167181
fund_age_years,9.283180,10.419109,1.135929
is_equity,0.385185,0.383392,-0.001793
is_fixed_income,0.259259,0.314488,0.055228


Table 2. Accessibility and Dividend Policies (Baseline)

In [80]:
import statsmodels.formula.api as smf

# Ensure there are no issues with NaNs in the group
df_model = df.dropna(subset=[
    'dividend',
    'retail_eligible_i',
    'CL4_yes',
    'Management_Group'
]).copy()

df_model['Management_Group_clean'] = df_model['Management_Group'].astype(str)

# Model with clustering
model = smf.ols(
    formula="dividend ~ retail_eligible_i + CL4_yes",
    data=df_model
).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_model['Management_Group_clean']}
)

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               dividend   R-squared:                       0.038
Model:                            OLS   Adj. R-squared:                  0.037
Method:                 Least Squares   F-statistic:                     17.72
Date:                Mon, 04 May 2026   Prob (F-statistic):           6.25e-08
Time:                        06:52:05   Log-Likelihood:                -1321.5
No. Observations:                1963   AIC:                             2649.
Df Residuals:                    1960   BIC:                             2666.
Df Model:                           2                                         
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -0.0568      0.09

Table 3. Fund Characteristics and Dividend Policies.

In [81]:
import statsmodels.formula.api as smf

df_model = df.dropna(subset=[
    'dividend',
    'retail_eligible_i',
    'CL4_yes',
    'log_tna',
    'fund_age_years',
    'is_equity',
    'is_fixed_income',
    'Management_Group'
]).copy()

df_model['Management_Group_clean'] = df_model['Management_Group'].astype(str)

model_controls = smf.ols(
    formula="""
    dividend ~ retail_eligible_i + CL4_yes
    + log_tna + fund_age_years
    + is_equity + is_fixed_income
    """,
    data=df_model
).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_model['Management_Group_clean']}
)

print(model_controls.summary())

                            OLS Regression Results                            
Dep. Variable:               dividend   R-squared:                       0.106
Model:                            OLS   Adj. R-squared:                  0.103
Method:                 Least Squares   F-statistic:                     14.72
Date:                Mon, 04 May 2026   Prob (F-statistic):           2.03e-14
Time:                        06:52:05   Log-Likelihood:                -1223.2
No. Observations:                1930   AIC:                             2460.
Df Residuals:                    1923   BIC:                             2499.
Df Model:                           6                                         
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -0.2605      0.11

Table 4. Within-Management Analysis.

In [82]:
model_fe = smf.ols(
    formula="""
    dividend ~ retail_eligible_i + CL4_yes
    + log_tna + fund_age_years
    + is_equity + is_fixed_income
    + C(Management_Group)
    """,
    data=df
).fit()

print(model_fe.summary())

                            OLS Regression Results                            
Dep. Variable:               dividend   R-squared:                       0.552
Model:                            OLS   Adj. R-squared:                  0.482
Method:                 Least Squares   F-statistic:                     7.941
Date:                Mon, 04 May 2026   Prob (F-statistic):          5.17e-162
Time:                        06:52:06   Log-Likelihood:                -556.32
No. Observations:                1930   AIC:                             1633.
Df Residuals:                    1670   BIC:                             3080.
Df Model:                         259                                         
Covariance Type:            nonrobust                                         
                                                   coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------

Table 5. Retail Targeting, Asset Class Interaction, and Management Effects

Without FE

In [83]:
df['Management_Group_clean'] = df['Management_Group'].fillna('Missing').astype(str)

In [84]:
import statsmodels.formula.api as smf

# 1. Required variables ONLY
vars_t5 = [
    'dividend',
    'retail_eligible_i',
    'is_fixed_income',
    'Management_Group_clean'
]

# 2. ONE SAMPLE FOR EVERYTHING
df_t5 = df[vars_t5].dropna()

# 3. Model
model_t5 = smf.ols(
    formula="dividend ~ retail_eligible_i * is_fixed_income",
    data=df_t5
).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_t5['Management_Group_clean']}
)

print(model_t5.summary())

                            OLS Regression Results                            
Dep. Variable:               dividend   R-squared:                       0.038
Model:                            OLS   Adj. R-squared:                  0.036
Method:                 Least Squares   F-statistic:                     14.93
Date:                Mon, 04 May 2026   Prob (F-statistic):           5.50e-09
Time:                        06:52:06   Log-Likelihood:                -1325.5
No. Observations:                1968   AIC:                             2659.
Df Residuals:                    1964   BIC:                             2681.
Df Model:                           3                                         
Covariance Type:              cluster                                         
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Interc

With FE

In [85]:
import statsmodels.formula.api as smf

# 1. Cleaned cluster variable
df['Management_Group_clean'] = df['Management_Group'].astype('category').cat.codes

# 2. Required variables
vars_t5 = [
    'dividend',
    'retail_eligible_i',
    'is_fixed_income',
    'Management_Group',
    'Management_Group_clean'
]

# 3. One size fits all
df_t5 = df[vars_t5].dropna()

# 4. Model with FE
model = smf.ols(
    formula="dividend ~ retail_eligible_i * is_fixed_income + C(Management_Group)",
    data=df_t5
).fit()

# 5. Aligned cluster
model_t5 = model.get_robustcov_results(
    cov_type='cluster',
    groups=df_t5.loc[model.model.data.row_labels, 'Management_Group_clean']
)

print(model_t5.summary())

                            OLS Regression Results                            
Dep. Variable:               dividend   R-squared:                       0.533
Model:                            OLS   Adj. R-squared:                  0.462
Method:                 Least Squares   F-statistic:                     105.2
Date:                Mon, 04 May 2026   Prob (F-statistic):           2.80e-44
Time:                        06:52:06   Log-Likelihood:                -612.62
No. Observations:                1963   AIC:                             1741.
Df Residuals:                    1705   BIC:                             3181.
Df Model:                         257                                         
Covariance Type:              cluster                                         
                                                   coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 257, but rank is 3
  warnings.warn('covariance of constraints does not have full '


Table 6. Retail Targeting vs Economic Accessibility.

Without FE

In [86]:
vars_t6 = [
    'dividend',
    'retail_eligible_i',
    'log_min_inv',
    'Management_Group_clean'
]

df_t6 = df[vars_t6].dropna()

model = smf.ols(
    formula="dividend ~ retail_eligible_i + log_min_inv",
    data=df_t6
).fit()

model_t6 = model.get_robustcov_results(
    cov_type='cluster',
    groups=df_t6.loc[model.model.data.row_labels, 'Management_Group_clean']
)

print(model_t6.summary())

                            OLS Regression Results                            
Dep. Variable:               dividend   R-squared:                       0.034
Model:                            OLS   Adj. R-squared:                  0.033
Method:                 Least Squares   F-statistic:                     15.70
Date:                Mon, 04 May 2026   Prob (F-statistic):           3.82e-07
Time:                        06:52:06   Log-Likelihood:                -1266.4
No. Observations:                1855   AIC:                             2539.
Df Residuals:                    1852   BIC:                             2555.
Df Model:                           2                                         
Covariance Type:              cluster                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             0.1876      0.08

With FE

In [87]:
vars_t6_fe = [
    'dividend',
    'retail_eligible_i',
    'log_min_inv',
    'Management_Group',
    'Management_Group_clean'
]

df_t6_fe = df[vars_t6_fe].dropna()

model = smf.ols(
    formula="dividend ~ retail_eligible_i + log_min_inv + C(Management_Group)",
    data=df_t6_fe
).fit()

model_t6_fe = model.get_robustcov_results(
    cov_type='cluster',
    groups=df_t6_fe.loc[model.model.data.row_labels, 'Management_Group_clean']
)

print(model_t6_fe.summary())

                            OLS Regression Results                            
Dep. Variable:               dividend   R-squared:                       0.542
Model:                            OLS   Adj. R-squared:                  0.471
Method:                 Least Squares   F-statistic:                     58.09
Date:                Mon, 04 May 2026   Prob (F-statistic):           2.17e-21
Time:                        06:52:06   Log-Likelihood:                -573.71
No. Observations:                1853   AIC:                             1645.
Df Residuals:                    1604   BIC:                             3021.
Df Model:                         248                                         
Covariance Type:              cluster                                         
                                                   coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 248, but rank is 2
  warnings.warn('covariance of constraints does not have full '


Table A.1 – Sample Comparison

In [88]:
df_full = pd.read_excel("ucits_dividend_data.xlsx")

vars_model = [
    'dividend_paid',
    'retail_eligible_i',
    'log_tna',
    'fund_age_years',
    'is_equity',
    'is_fixed_income'
]

df_final = df_full[vars_model].dropna()

# -----------------------------
# SAMPLE COMPARISON
# -----------------------------
vars_compare = ['log_tna', 'fund_age_years', 'is_equity', 'is_fixed_income']

summary_full = df_full[vars_compare].mean()
summary_final = df_final[vars_compare].mean()

comparison = pd.DataFrame({
    'Full sample': summary_full,
    'Final sample': summary_final,
    'Difference': summary_final - summary_full
})

print(comparison)

                 Full sample  Final sample  Difference
log_tna             3.929630      4.162036    0.232406
fund_age_years     14.159420     10.104643   -4.054777
is_equity           0.348268      0.386047    0.037779
is_fixed_income     0.299111      0.309044    0.009933


Table A.2. Alternative Measure of Economic Accessibility

In [89]:
# Subsample
vars_model = [
    'dividend_paid',
    'retail_eligible_i',
    'high_min_inv',
    'log_tna',
    'fund_age_years',
    'is_equity',
    'is_fixed_income',
    'Management_Group'
]

df_base = df[vars_model].dropna()

# Model
model_alt = smf.ols(
    "dividend_paid ~ retail_eligible_i + high_min_inv + log_tna + fund_age_years + is_equity + is_fixed_income",
    data=df_base
).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_base['Management_Group']}
)

# 5. Output
print(model_alt.summary())

                            OLS Regression Results                            
Dep. Variable:          dividend_paid   R-squared:                       0.071
Model:                            OLS   Adj. R-squared:                  0.068
Method:                 Least Squares   F-statistic:                     4.198
Date:                Mon, 04 May 2026   Prob (F-statistic):           0.000479
Time:                        06:52:08   Log-Likelihood:                -859.84
No. Observations:                1930   AIC:                             1734.
Df Residuals:                    1923   BIC:                             1773.
Df Model:                           6                                         
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             0.0441      0.04

Table A.3. Dividend Intensity and Retail Targeting

In [90]:
# Controls
controls = "log_tna + fund_age_years + is_equity + is_fixed_income"

vars_base = [
    'dividend_paid',
    'retail_eligible_i',
    'log_tna',
    'fund_age_years',
    'is_equity',
    'is_fixed_income',
    'Management_Group'
]

df_base = df[vars_base].dropna()

# -----------------------------
# (1) BASELINE MODEL
# -----------------------------
model1 = smf.ols(
    f"dividend_paid ~ retail_eligible_i + {controls}",
    data=df_base
).fit()

res1 = model1.get_robustcov_results(
    cov_type='cluster',
    groups=df_base.loc[model1.model.data.row_labels, 'Management_Group']
)

# -----------------------------
# (2) INTENSITY (PAYING MEMBERS ONLY)
# -----------------------------
df_paid = df_base.copy()
df_paid['log_div_value'] = df['log_div_value']

df_paid = df_paid[
    (df_paid['dividend_paid'] == 1) &
    (df_paid['log_div_value'].notnull())
]

model2 = smf.ols(
    f"log_div_value ~ retail_eligible_i + {controls}",
    data=df_paid
).fit()

res2 = model2.get_robustcov_results(
    cov_type='cluster',
    groups=df_paid.loc[model2.model.data.row_labels, 'Management_Group']
)

# Output
print("\n(1) Dividend (Baseline)")
print(res1.summary())

print("\n(2) Log Dividend Value (Paid Funds)")
print(res2.summary())


(1) Dividend (Baseline)
                            OLS Regression Results                            
Dep. Variable:          dividend_paid   R-squared:                       0.064
Model:                            OLS   Adj. R-squared:                  0.062
Method:                 Least Squares   F-statistic:                     4.956
Date:                Mon, 04 May 2026   Prob (F-statistic):           0.000239
Time:                        06:52:09   Log-Likelihood:                -867.42
No. Observations:                1930   AIC:                             1747.
Df Residuals:                    1924   BIC:                             1780.
Df Model:                           5                                         
Covariance Type:              cluster                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept    